# 🎙️ F5-Hindi TTS — SPRINGLab/F5-Hindi-24KHz
> Zero-shot Hindi Text-to-Speech powered by F5-TTS (Flow Matching)  
> Run each cell **top-to-bottom** in Google Colab. A **GPU runtime** is strongly recommended.

**Workflow:**
1. ✅ Check GPU
2. 📦 Install dependencies
3. ⬇️  Download model from HuggingFace
4. 📤 Upload your Hindi text file
5. ⚙️  Configure voice & generation settings
6. 🔊 Generate speech
7. 💾 Download the audio file

## Cell 1 — GPU Check
Verify that a GPU is available. If not, go to **Runtime → Change runtime type → GPU**.

In [ ]:
import subprocess, sys

# Check for GPU
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print('✅ GPU detected!')
    print(result.stdout)
else:
    print('⚠️  No GPU found. Go to Runtime → Change runtime type → GPU.')

import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


## Cell 2 — Install Dependencies
Installs **f5-tts** (PyPI release — supports `model_cfg` required for the SPRINGLab Hindi checkpoint)  
and `huggingface_hub` for model downloading. This may take **3–5 minutes**.

In [ ]:
import subprocess, sys

print('📦 Installing F5-TTS and dependencies...')
print('   This takes 2–4 minutes on a fresh Colab runtime.\n')

# Install f5-tts from PyPI (includes all required dependencies)
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'f5-tts',
    'soundfile',
    'cached_path',
    'huggingface_hub',
])

# Verify core imports work safely
try:
    from f5_tts.model.backbones.dit import DiT
    from f5_tts.infer.utils_infer import load_model, load_vocoder, infer_process
    print(f'\n✅ f5-tts installed and verified')
    print('   ✓ Core modules successfully imported')
    print('   All dependencies ready.')
except Exception as e:
    print(f'\n❌ Installation issue: {e}')


## Cell 3 — Download Model from HuggingFace
Downloads `model_2500000.safetensors`, `vocab.txt`, and bundled **reference audio samples** from  
[SPRINGLab/F5-Hindi-24KHz](https://huggingface.co/SPRINGLab/F5-Hindi-24KHz).

In [ ]:
from huggingface_hub import hf_hub_download, snapshot_download
import os

MODEL_REPO = 'SPRINGLab/F5-Hindi-24KHz'
MODEL_DIR  = '/content/F5-Hindi-24KHz'
os.makedirs(MODEL_DIR, exist_ok=True)

print(f'⬇️  Downloading model from {MODEL_REPO} ...')

# Download model checkpoint
ckpt_path = hf_hub_download(
    repo_id=MODEL_REPO,
    filename='model_2500000.safetensors',
    local_dir=MODEL_DIR,
)
print(f'  ✓ Checkpoint : {ckpt_path}')

# Download vocabulary file
vocab_path = hf_hub_download(
    repo_id=MODEL_REPO,
    filename='vocab.txt',
    local_dir=MODEL_DIR,
)
print(f'  ✓ Vocabulary : {vocab_path}')

# Download bundled reference audio samples
sample_files = [
    'samples/output1.wav',
    'samples/output2.wav',
    'samples/output3.wav',
]
downloaded_samples = []
for sf in sample_files:
    try:
        sp = hf_hub_download(repo_id=MODEL_REPO, filename=sf, local_dir=MODEL_DIR)
        downloaded_samples.append(sp)
        print(f'  ✓ Sample     : {sp}')
    except Exception as e:
        print(f'  ⚠ Could not download {sf}: {e}')

print(f'\n✅ Model downloaded to: {MODEL_DIR}')
print(f'   Available reference samples: {len(downloaded_samples)}')


## Cell 4 — Upload Your Hindi Text File
Upload a **`.txt` file** containing the Hindi text you want to convert to speech.  
> The file should be saved in **UTF-8** encoding and contain plain Hindi (Devanagari script).

In [ ]:
from google.colab import files
import os

print('📤 Please upload your Hindi text file (.txt)...')
uploaded = files.upload()

if not uploaded:
    raise ValueError('No file uploaded. Please re-run this cell and upload a .txt file.')

# Get the first uploaded file
txt_filename = list(uploaded.keys())[0]
if not txt_filename.endswith('.txt'):
    print(f'⚠️  Warning: file "{txt_filename}" does not have a .txt extension.')

# Decode content
gen_text = uploaded[txt_filename].decode('utf-8').strip()

print(f'\n✅ File loaded: {txt_filename}')
print(f'   Characters : {len(gen_text)}')
print(f'   Words      : {len(gen_text.split())}')
print(f'\n--- Text Preview (first 300 chars) ---')
print(gen_text[:300])
if len(gen_text) > 300:
    print('...[truncated]')


## Cell 5 — ⚙️ Configure Voice & Generation Settings
**Edit the variables in this cell** to customise the output.  

| Parameter | Description |
|-----------|-------------|
| `VOICE_CHOICE` | Reference speaker audio (1, 2, or 3; or `'custom'` to upload your own) |
| `SPEED` | Speech speed — `1.0` = normal, `<1` = slower, `>1` = faster |
| `NFE_STEPS` | Diffusion steps — more = better quality but slower (16–64) |
| `CFG_STRENGTH` | Classifier-free guidance strength — higher = closer to reference style (1–4) |
| `SWAY_COEF` | Sway sampling coefficient — improves quality (`-1` = auto, or `0.0`–`1.0`) |
| `OUTPUT_FILENAME` | Name of the generated WAV file |
| `REMOVE_SILENCE` | Strip leading/trailing silence from output |

> **Voice Selection:** The model uses *zero-shot voice cloning* — it copies the speaking style  
> from a short reference audio clip. Choose a bundled sample or supply your own 5–15 sec WAV.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║              🎛️  USER CONFIGURATION — EDIT HERE              ║
# ╚══════════════════════════════════════════════════════════════╝

# ── Voice (Reference Audio) ────────────────────────────────────
# Choose 1, 2, or 3 to use a bundled Hindi speaker sample,
# or set to 'custom' to upload your own reference audio file.
VOICE_CHOICE = 1          # Options: 1 | 2 | 3 | 'custom'

# ── Speech Speed ───────────────────────────────────────────────
# 1.0 = normal, 0.8 = 20% slower, 1.2 = 20% faster
SPEED = 1.0               # Range: 0.5 – 2.0

# ── Diffusion Quality ──────────────────────────────────────────
# Higher NFE = better quality but slower inference
NFE_STEPS = 32            # Recommended: 16 (fast) | 32 (balanced) | 64 (high quality)

# ── Guidance Strength ──────────────────────────────────────────
# Controls how closely the model follows the reference voice style
CFG_STRENGTH = 2.0        # Range: 1.0 – 4.0  (default: 2.0)

# ── Sway Sampling ──────────────────────────────────────────────
# Inference-time flow step strategy; improves naturalness
SWAY_COEF = -1.0          # -1.0 = auto | try 0.0–1.0 to tune

# ── Output File ────────────────────────────────────────────────
OUTPUT_FILENAME = 'hindi_tts_output.wav'

# ── Silence Removal ────────────────────────────────────────────
REMOVE_SILENCE = False    # True to strip leading/trailing silence

# ╔══════════════════════════════════════════════════════════════╗
# ║                END OF USER CONFIGURATION                     ║
# ╚══════════════════════════════════════════════════════════════╝

import os
from google.colab import files as colab_files

MODEL_DIR = '/content/F5-Hindi-24KHz'
OUTPUT_DIR = '/content/tts_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Reference Audio texts (matching the bundled samples) ───────
# These transcripts correspond to the downloaded .wav files.
SAMPLE_REF_TEXTS = {
    1: '',
    2: '',
    3: '',
}

SAMPLE_AUDIO_PATHS = {
    1: os.path.join(MODEL_DIR, 'samples', 'output1.wav'),
    2: os.path.join(MODEL_DIR, 'samples', 'output2.wav'),
    3: os.path.join(MODEL_DIR, 'samples', 'output3.wav'),
}

# ── Resolve voice selection ─────────────────────────────────────
if VOICE_CHOICE == 'custom':
    print('📤 Upload your custom reference audio file (WAV, 5–15 sec, 24kHz preferred)...')
    ref_uploaded = colab_files.upload()
    if not ref_uploaded:
        raise ValueError('No reference audio uploaded.')
    ref_name = list(ref_uploaded.keys())[0]
    ref_audio_path = f'/content/{ref_name}'
    with open(ref_audio_path, 'wb') as f:
        f.write(ref_uploaded[ref_name])
    ref_text = input('Enter the transcript of your reference audio (in Hindi): ').strip()
    print(f'✅ Custom reference audio set: {ref_audio_path}')
elif VOICE_CHOICE in SAMPLE_AUDIO_PATHS:
    ref_audio_path = SAMPLE_AUDIO_PATHS[VOICE_CHOICE]
    ref_text = SAMPLE_REF_TEXTS[VOICE_CHOICE]
    if not os.path.exists(ref_audio_path):
        raise FileNotFoundError(
            f'Reference audio not found: {ref_audio_path}\n'
            f'Make sure Cell 3 ran successfully and the file downloaded.'
        )
    print(f'✅ Voice        : Sample {VOICE_CHOICE}')
    print(f'   Audio file   : {ref_audio_path}')
    print(f'   Ref text     : {ref_text[:80]}...')
else:
    raise ValueError(f'Invalid VOICE_CHOICE: {VOICE_CHOICE!r}. Use 1, 2, 3, or "custom".')

print(f'\n🎛️  Generation Settings:')
print(f'   Speed          : {SPEED}')
print(f'   NFE Steps      : {NFE_STEPS}')
print(f'   CFG Strength   : {CFG_STRENGTH}')
print(f'   Sway Coef      : {SWAY_COEF}')
print(f'   Remove Silence : {REMOVE_SILENCE}')
print(f'   Output file    : {OUTPUT_DIR}/{OUTPUT_FILENAME}')
print('\n✅ Configuration complete. Ready to generate!')


## Cell 6 — Generate Speech
Loads the model and synthesises speech from your text.  
> For long texts the model will automatically chunk and concatenate segments.

In [ ]:
import torch
import soundfile as sf
import numpy as np
import os, time
from IPython.display import Audio, display

# ── Device selection ───────────────────────────────────────────
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🖥️  Running on: {device.upper()}')
if device == 'cpu':
    print('   ⚠️  CPU mode is very slow. Consider enabling GPU in Runtime settings.')

# ── Model configuration (F5-TTS Hindi "small" architecture) ───
MODEL_ARCH = {
    'dim': 768,
    'depth': 18,
    'heads': 12,
    'ff_mult': 2,
    'text_dim': 512,
    'conv_layers': 4,
}

CKPT_FILE  = os.path.join(MODEL_DIR, 'model_2500000.safetensors')
VOCAB_FILE = os.path.join(MODEL_DIR, 'vocab.txt')

# Validate files exist
for fpath, label in [(CKPT_FILE, 'Checkpoint'), (VOCAB_FILE, 'Vocabulary'), (ref_audio_path, 'Reference audio')]:
    if not os.path.exists(fpath):
        raise FileNotFoundError(f'{label} not found: {fpath}. Did earlier cells run successfully?')

# ── Load model using low-level API ─────────────────────────────
print('⏳ Loading F5-TTS Hindi model...')
t0 = time.time()

from f5_tts.model.backbones.dit import DiT
from f5_tts.infer.utils_infer import (
    load_model,
    load_vocoder,
    preprocess_ref_audio_text,
    infer_process,
)

# Load the vocoder
mel_spec_type = 'vocos'
vocoder = load_vocoder(mel_spec_type, False, None, device)

# Load the Hindi model
ema_model = load_model(
    model_cls=DiT,
    model_cfg=MODEL_ARCH,
    ckpt_path=CKPT_FILE,
    mel_spec_type=mel_spec_type,
    vocab_file=VOCAB_FILE,
    ode_method='euler',
    use_ema=True,
    device=device,
)
print(f'✅ Model loaded in {time.time() - t0:.1f}s')

# ── Preprocess reference audio ─────────────────────────────────
ref_audio_processed, ref_text_processed = preprocess_ref_audio_text(
    ref_audio_path, ref_text
)

# ── Inference ─────────────────────────────────────────────────
print('\n🔊 Generating speech...')
print(f'   Text length : {len(gen_text)} characters')
t1 = time.time()

wav, sample_rate, spec = infer_process(
    ref_audio_processed,
    ref_text_processed,
    gen_text,
    ema_model,
    vocoder,
    mel_spec_type,
    nfe_step=NFE_STEPS,
    cfg_strength=CFG_STRENGTH,
    sway_sampling_coef=SWAY_COEF,
    speed=SPEED,
    device=device,
)

elapsed = time.time() - t1
duration = len(wav) / sample_rate
print(f'✅ Speech generated in {elapsed:.1f}s')
print(f'   Audio duration : {duration:.2f}s')

# ── Save output ───────────────────────────────────────────────
output_path = os.path.join(OUTPUT_DIR, OUTPUT_FILENAME)
wav_np = np.array(wav, dtype=np.float32)
if wav_np.ndim > 1:
    wav_np = wav_np.squeeze()

sf.write(output_path, wav_np, sample_rate)
print(f'\n💾 Saved to: {output_path}')

# Optionally remove silence
if REMOVE_SILENCE:
    from f5_tts.infer.utils_infer import remove_silence_for_generated_wav
    remove_silence_for_generated_wav(output_path)

# ── Playback in notebook ───────────────────────────────────────
print('\n🎧 Playing generated audio:')
display(Audio(wav_np, rate=sample_rate, autoplay=True))

## Cell 7 — Download Audio
Downloads the generated WAV file to your local machine.

In [ ]:
from google.colab import files as colab_files
import os

output_path = os.path.join(OUTPUT_DIR, OUTPUT_FILENAME)

if not os.path.exists(output_path):
    raise FileNotFoundError(
        f'Output file not found: {output_path}\n'
        'Make sure Cell 6 ran successfully before downloading.'
    )

file_size_kb = os.path.getsize(output_path) / 1024
print(f'📁 File        : {OUTPUT_FILENAME}')
print(f'   Size        : {file_size_kb:.1f} KB')
print(f'   Path        : {output_path}')
print('\n⬇️  Starting download...')

colab_files.download(output_path)
print('✅ Download initiated. Check your browser downloads folder.')


---
## Optional — Re-generate with Custom Text (No File Upload)
Type Hindi text directly in the cell below and run it to regenerate audio  
without re-loading the model.

In [ ]:
# ── Quick regeneration cell ───────────────────────────────────
# Edit the text below and run this cell to generate a new clip.
# The model stays loaded — no need to re-run earlier cells.

custom_text = """नमस्ते! मैं हिंदी में बोल रहा हूँ। यह एक परीक्षण वाक्य है।"""
custom_speed = SPEED

import soundfile as sf
import numpy as np
import time
from IPython.display import Audio, display
from f5_tts.infer.utils_infer import preprocess_ref_audio_text, infer_process

print('🔊 Generating speech for custom text...')
t_start = time.time()
ref_audio_proc, ref_text_proc = preprocess_ref_audio_text(ref_audio_path, ref_text)

wav2, sr2, _ = infer_process(
    ref_audio_proc,
    ref_text_proc,
    custom_text.strip(),
    ema_model,
    vocoder,
    mel_spec_type,
    nfe_step=NFE_STEPS,
    cfg_strength=CFG_STRENGTH,
    sway_sampling_coef=SWAY_COEF,
    speed=custom_speed,
    device=device,
)

wav2 = np.array(wav2, dtype=np.float32).squeeze()
custom_out = '/content/tts_output/custom_output.wav'
sf.write(custom_out, wav2, sr2)

print(f'✅ Done in {time.time() - t_start:.1f}s | Duration: {len(wav2)/sr2:.2f}s')
display(Audio(wav2, rate=sr2, autoplay=True))

from google.colab import files as colab_files
colab_files.download(custom_out)